In [2]:

# -*- coding: utf-8 -*-
"""
Envio de e-mails por Coordenador + Fornecedor com atualização da base no Excel.
Requisitos:
- pandas
- pywin32 (win32com)
- xlwings

Observações:
- Execute com o Outlook aberto (perfil padrão).
- Ajuste o caminho da base, se necessário.
"""

import sys
import pandas as pd
import win32com.client as win32
import xlwings as xw

# ================================================================
# CONFIGURAÇÕES INICIAIS
# ================================================================

base_original = r"C:\Users\126815\OneDrive - paguemenos.com.br\ENGENHARIA - OBRAS - Documentos\BANCO DE DADOS - POWER BI\BI - OUTROS\BASE CONTROLE DE PAGAMENTOS.xlsx"

print("📂 Lendo a aba 'SOLICITAÇÃO DE PAGAMENTO' da base...")
df = pd.read_excel(base_original, sheet_name="SOLICITAÇÃO DE PAGAMENTO")
print(f"✅ Base carregada com sucesso! Linhas totais: {len(df)}\n")

# ================================================================
# NORMALIZAÇÃO DAS COLUNAS
# ================================================================

df.columns = df.columns.str.replace("\xa0", " ").str.strip().str.upper()

# Encontra a coluna "ENVIAR COORD?"
possiveis = [col for col in df.columns if "ENVIAR COORD" in col]
if not possiveis:
    raise ValueError("❌ Coluna 'ENVIAR COORD?' não encontrada na base!")
coluna_coord = possiveis[0]
print(f"🔎 Coluna de controle encontrada: '{coluna_coord}'")

# ================================================================
# FILTRAGEM DOS DADOS
# ================================================================

# ✅ Correção: usar .str.strip() e não .strip() direto
df_filtrado = df[df[coluna_coord].astype(str).str.strip().str.upper() == "SIM"]

colunas_exibicao = ['FILIAL', 'LOJA', 'CNPJ', 'COORDENADOR', 'PROJETO', 'SERVIÇO', 'FORNECEDOR', 'VALOR RC', 'PEDIDO']
colunas_existentes = [c for c in colunas_exibicao if c in df.columns]

df_filtrado = df_filtrado[colunas_existentes].copy()

# Formatação de valor (quando existir e for numérico)
if 'VALOR RC' in df_filtrado.columns:
    df_filtrado['VALOR RC'] = df_filtrado['VALOR RC'].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        if pd.notnull(x) and isinstance(x, (int, float)) else x
    )

# ================================================================
# E-MAILS DOS COORDENADORES
# ================================================================

emails_coordenadores = {
    'ADRIANO': 'carolinerocha@pmenos.com.br',
    'ANDRE': 'luizfrota@pmenos.com.br',
    'HENRIQUE': 'henriquesales@pmenos.com.br',
    'JORDANA': 'jordanasilva@pmenos.com.br',
    'CAROL': 'carolinerocha@pmenos.com.br'
}

# ================================================================
# FUNÇÃO DE ENVIO DE E-MAIL
# ================================================================

def enviar_email_outlook(coordenador: str, fornecedor: str, dados: pd.DataFrame, colunas_para_mostrar):
    email_destino = emails_coordenadores.get(coordenador.upper())
    if not email_destino:
        print(f"⚠️ Coordenador sem e-mail definido: {coordenador}")
        return

    # Monta tabela HTML
    thead = "".join(f"<th>{col.title()}</th>" for col in colunas_para_mostrar)

    linhas_html = []
    for _, linha in dados.iterrows():
        tds = []
        for col in colunas_para_mostrar:
            valor = "" if pd.isna(linha.get(col)) else linha[col]
            tds.append(f"<td>{valor}</td>")
        linhas_html.append(f"<tr>{''.join(tds)}</tr>")

    tbody_html = "\n".join(linhas_html)

    corpo_html = f"""
    <html>
    <head>
      <style>
        body {{ font-family: Calibri, sans-serif; font-size: 13px; color: #333; padding: 10px; }}
        table {{ border-collapse: collapse; width: 100%; margin-top: 15px; border: 1px solid #ddd; }}
        th {{ background-color: #f3f3f3; border: 1px solid #ddd; text-align: center; padding: 8px; font-weight: bold; }}
        td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        td:nth-child(7), td:nth-child(8) {{ text-align: right; }}
        tr:nth-child(even) {{ background-color: #fafafa; }}
        tr:hover {{ background-color: #f1f7ff; }}
      </style>
    </head>
    <body>
      <p>Olá {coordenador},</p>
      <p>Segue abaixo a relação de pedidos criados para o fornecedor <strong>{fornecedor}</strong>:</p>
      <table>
        <thead><tr>{thead}</tr></thead>
        <tbody>
          {tbody_html}
        </tbody>
      </table>
      <p>Por favor, verifique os pedidos e siga com as tratativas necessárias.</p>
      <br>
      <p>
        Atenciosamente,<br>
        <strong>Tiago Felix de Oliveira</strong><br>
        Analista Administrativo I<br>
        Pague Menos
      </p>
    </body>
    </html>
    """

    mail = outlook.CreateItem(0)
    mail.To = email_destino
    mail.CC = "tiagooliveira@pmenos.com.br;jordanasilva@pmenos.com.br"
    mail.Subject = f"Pedidos Criados - Fornecedor {fornecedor} - Coord {coordenador}"
    mail.HTMLBody = corpo_html
    mail.Send()

    print(f"✅ E-mail enviado para {coordenador} ({email_destino}) - Fornecedor: {fornecedor}")

# ================================================================
# EXECUÇÃO DOS ENVIOS
# ================================================================

agrupamento = []
if not df_filtrado.empty:
    faltam = [c for c in ['COORDENADOR', 'FORNECEDOR'] if c not in df_filtrado.columns]
    if faltam:
        print(f"❌ Colunas ausentes para agrupamento: {', '.join(faltam)}")
        sys.exit(1)
    agrupado = df_filtrado.groupby(['COORDENADOR', 'FORNECEDOR'])
    agrupamento = list(agrupado)

print("\n📋 Linhas filtradas que serão enviadas:")
if df_filtrado.empty:
    print("Nenhum registro encontrado para envio.")
else:
    print(df_filtrado.to_string(index=False))

print(f"\n📊 Foram encontrados {len(df_filtrado)} registros filtrados.")
print(f"📬 Serão enviados {len(agrupamento)} e-mails (agrupados por Coordenador + Fornecedor).")

# Outlook
try:
    outlook = win32.Dispatch('outlook.application')
except Exception as e:
    print(f"❌ Não foi possível conectar ao Outlook: {e}")
    sys.exit(1)

resposta = input("\nDeseja enviar os e-mails agora? (s/n): ").strip().lower()
if resposta == 's' and agrupamento:
    app = xw.App(visible=False, add_book=False)
    try:
        wb = app.books.open(base_original)
        ws = wb.sheets["SOLICITAÇÃO DE PAGAMENTO"]

        # Cabeçalho (linha 1) para localizar o índice da coluna de controle
        cabecalho = ws.range("A1").expand("right").value
        try:
            col_idx = cabecalho.index(coluna_coord) + 1
        except ValueError:
            print(f"❌ Coluna '{coluna_coord}' não encontrada no Excel.")
            wb.close()
            app.quit()
            sys.exit(1)

        # Envio por grupo e marcação "ENVIADO"
        for (coordenador, fornecedor), grupo in agrupamento:
            enviar_email_outlook(coordenador, fornecedor, grupo, colunas_existentes)
            for i in grupo.index:
                ws.cells(i + 2, col_idx).value = "ENVIADO"  # A1 é cabeçalho → dados iniciam na linha 2

        wb.save()
        print("\n✅ Todos os e-mails foram enviados e a base atualizada com 'ENVIADO'.")
    finally:
        app.quit()
else:
    print("❌ Envio cancelado pelo usuário ou não há grupos para enviar.")


📂 Lendo a aba 'SOLICITAÇÃO DE PAGAMENTO' da base...
✅ Base carregada com sucesso! Linhas totais: 13365

🔎 Coluna de controle encontrada: 'ENVIAR COORD?'

📋 Linhas filtradas que serão enviadas:
FILIAL                 LOJA                       CNPJ COORDENADOR            PROJETO      SERVIÇO FORNECEDOR      VALOR RC     PEDIDO
  7037 SLU41-AlcMachA-MA-EF         04.899.316/0037-29    HENRIQUE VIRADA DE BANDEIRA Gerenciadora    METROLL   R$ 5.400,00 4501113646
  7038      SLU43-203-MA-EF         04.899.316/0038-00    HENRIQUE VIRADA DE BANDEIRA Gerenciadora    METROLL   R$ 5.400,00 4501113647
  7042 SLU46-LesOes18-MA-EF         04.899.316/0042-96    HENRIQUE VIRADA DE BANDEIRA Gerenciadora    METROLL   R$ 5.400,00 4501113648
  7044 SLU40-ColMor44-MA-EF         04.899.316/0044-58    HENRIQUE VIRADA DE BANDEIRA Gerenciadora    METROLL   R$ 5.400,00 4501113649
  7051 SLU44-DanLaTou-MA-EF         04.899.316/0051-87    HENRIQUE VIRADA DE BANDEIRA Gerenciadora    METROLL   R$ 5.400,00 45011136


Deseja enviar os e-mails agora? (s/n):  s


✅ E-mail enviado para ANDRE (luizfrota@pmenos.com.br) - Fornecedor: DWA
✅ E-mail enviado para CAROL (carolinerocha@pmenos.com.br) - Fornecedor: PLANAC
✅ E-mail enviado para HENRIQUE (henriquesales@pmenos.com.br) - Fornecedor: METROLL

✅ Todos os e-mails foram enviados e a base atualizada com 'ENVIADO'.
